# Prediction-Level Audit and Table Verification

This notebook creates the prediction-level audit artifacts requested for
independent verification. It reads the released test predictions, split
assignments, training weights, and LICE-pattern trace. It does not retrain any
model and does not rerun LIME or DiCE.

The automated verification reloads the released prediction file and
reconstructs confusion matrices, full-precision metrics, false-negative
transitions, McNemar results, and Tables 3-10.

## 1. Connect Google Drive and install the environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Research/"
    "LICE_Guided_Model_Refinement_Release_V3"
)
assert PROJECT_ROOT.exists(), f"Release folder not found: {PROJECT_ROOT}"

from importlib.metadata import PackageNotFoundError, version
import os
import sys

PINNED_BINARY_STACK = {
    "numpy": ("numpy", "2.5.1"),
    "pandas": ("pandas", "2.2.3"),
    "scikit-learn": ("sklearn", "1.5.2"),
    "scipy": ("scipy", "1.18.0"),
    "statsmodels": ("statsmodels", "0.14.4"),
}


def installed_version(distribution_name):
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return None


restart_required = any(
    installed_version(distribution) != required
    or (
        module in sys.modules
        and getattr(sys.modules[module], "__version__", None) != required
    )
    for distribution, (module, required) in PINNED_BINARY_STACK.items()
)

%pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"

if restart_required:
    print(
        "Pinned packages were installed. Colab will restart now. "
        "After it reconnects, run this setup cell once more and "
        "then continue to the next cell.",
        flush=True,
    )
    os.kill(os.getpid(), 9)

print(f"Release folder: {PROJECT_ROOT}")

## 2. Build the audit artifacts and run automated verification

The verification program is intentionally independent of the in-memory model
training workflow. A failure raises an exception and prevents a successful
completion message.

In [4]:
import runpy

verification_program = (
    PROJECT_ROOT / "tests" / "verify_released_predictions.py"
)

assert verification_program.exists(), verification_program

verification_namespace = runpy.run_path(
    str(verification_program),
    run_name="__main__",
)

del verification_namespace

                                                  Check_ID                                               Scope                                         Observed                                         Expected                                                      Tolerance Status
              input_exists_diabetes_brfss2015_prepared.csv                                  Input availability                                             True                                             True                                                          exact   PASS
                     input_exists_split_assignments.csv.gz                                  Input availability                                             True                                             True                                                          exact   PASS
                   input_exists_lice_sample_weights.csv.gz                                  Input availability                                             True    

## 3. Inspect the verification outputs

In [5]:
import json

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

results_dir = PROJECT_ROOT / "results"
checks = pd.read_csv(results_dir / "prediction_reproduction_checks.csv")
transitions = pd.read_csv(
    results_dir / "fn_transition_summary_from_predictions.csv"
)
with (results_dir / "prediction_audit_manifest.json").open(
    encoding="utf-8"
) as stream:
    manifest = json.load(stream)

assert checks["Status"].eq("PASS").all()
assert manifest["all_checks_passed"] is True
assert manifest["row_counts"]["test_instances"] == 13812
assert manifest["row_counts"]["model_variants"] == 8
assert manifest["row_counts"]["long_prediction_rows"] == 110496
assert manifest["row_counts"]["training_audit_rows"] == 55245

display(transitions)
display(checks[["Check_ID", "Scope", "Status"]])

print(
    f"Checks passed: {checks['Status'].eq('PASS').sum()} / {len(checks)}"
)
print("All prediction-level artifacts and Tables 3-10 were verified.")

,Model,FN_to_TP,TP_to_FN,FN_to_FN,TP_to_TP,Net_FN_Reduction
0,M1-Interactions,102,78,1315,5594,24
1,M2-Mild,122,31,1295,5641,91
2,M2-Balanced,176,24,1241,5648,152
3,M2-High,201,24,1216,5648,177
4,M3-Mild,161,56,1256,5616,105
5,M3-Balanced,212,40,1205,5632,172
6,M3-High,248,52,1169,5620,196


,Check_ID,Scope,Status
0,input_exists_diabetes_brfss2015_prepared.csv,Input availability,PASS
1,input_exists_split_assignments.csv.gz,Input availability,PASS
2,input_exists_lice_sample_weights.csv.gz,Input availability,PASS
3,input_exists_pattern_match_trace.csv.gz,Input availability,PASS
4,input_exists_predictions_all_models.csv.gz,Input availability,PASS
...,...,...,...
63,table_6_reproduction,Table 6,PASS
64,table_7_reproduction,Table 7,PASS
65,table_8_reproduction,Table 8,PASS
66,table_9_reproduction,Table 9,PASS


Checks passed: 68 / 68
All prediction-level artifacts and Tables 3-10 were verified.


## Interpretation of fold and sample-weight fields

The 13,812 evaluation observations belong to the untouched outer test set and
therefore do not have OOF validation-fold assignments. Their evaluation weight
is 1.0 for every model. The model-specific Mild, Balanced, and High-Sensitivity
weights apply only to training observations and are provided with OOF fold and
LICE-pattern fields in `results/training_lice_assignment_audit.csv.gz`.